In [1]:
"""
╔════════════════════════════════════════════════════════════════════════════╗
║  🎯 CELLULE DE CONFIGURATION STANDALONE - COPIER-COLLER DANS VOS NOTEBOOKS ║
╚════════════════════════════════════════════════════════════════════════════╝

INSTRUCTIONS:
-------------
1. Copiez TOUT le contenu de cette cellule
2. Collez-le comme PREMIÈRE CELLULE de votre notebook
3. Exécutez la cellule
4. La variable 'config' est prête à l'emploi !

Cette cellule est 100% autonome et fonctionne partout :
✅ Google Colab (clone + installe automatiquement)
✅ WSL / Linux Local
✅ Tout environnement Jupyter

APRÈS EXÉCUTION, VOUS POUVEZ UTILISER:
- config: Objet de configuration (config.batch_size, config.data_dir, etc.)
- ENV: Environnement détecté ('colab', 'wsl', 'local')
- from features.raf.*: Tous les imports du projet

"""

# =============================================================================
# IMPORTS STANDARDS
# =============================================================================

import os
import sys
import json
import subprocess
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional


# =============================================================================
# DÉTECTION AUTOMATIQUE DE L'ENVIRONNEMENT
# =============================================================================

def detect_environment():
    """Détecte l'environnement (colab, wsl, local)"""
    try:
        import google.colab
        return "colab"
    except ImportError:
        is_wsl = os.path.exists('/proc/version') and 'microsoft' in open('/proc/version').read().lower()
        return "wsl" if is_wsl else "local"

ENV = detect_environment()
print(f"🌍 Environnement: {ENV.upper()}")


# =============================================================================
# BOOTSTRAP COLAB (Clone + Install si nécessaire)
# =============================================================================

if ENV == "colab":
    print("\n🚀 Bootstrap Colab...")
    
    os.chdir('/content')
    if not os.path.exists('/content/DS_COVID'):
        print("📥 Clonage du repository...")
        subprocess.run(['git', 'clone', 'https://github.com/L-Poca/DS_COVID.git'], check=True)
    
    os.chdir('/content/DS_COVID')
    subprocess.run(['git', 'checkout', 'copilot/data-viz-exploratory-analysis-again'], 
                   capture_output=True, check=False)
    
    print("📦 Installation des dépendances...")
    subprocess.run(['pip', 'install', '-r', 'requirements-colab.txt', '--quiet'], check=True)
    
    print("📦 Installation du package...")
    result = subprocess.run(['pip', 'install', '-e', '.', '--quiet'], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"⚠️ Erreur installation: {result.stderr}")
    else:
        print("✅ Package installé")
    
    print("💾 Montage Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Extraction dataset
    for archive in ['/content/drive/MyDrive/DS_COVID/archive_covid.zip']:
        if os.path.exists(archive):
            print("📦 Extraction dataset...")
            os.makedirs('./data/raw/', exist_ok=True)
            subprocess.run(['unzip', '-o', '-q', archive, '-d', './data/raw/COVID-19_Radiography_Dataset/'])
            break
    
    print("✅ Bootstrap terminé")


# =============================================================================
# AJOUT DU CHEMIN src/ POUR LES IMPORTS
# =============================================================================

# Déterminer project_root selon l'environnement
if ENV == "colab":
    project_root = Path('/content/DS_COVID')
else:
    project_root = Path('/home/cepa/DST/projet_DS/DS_COVID')

# Ajouter src/ au sys.path pour permettre "from features.raf.*"
src_path = str(project_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    print(f"✅ Chemin src/ ajouté: {src_path}")


# =============================================================================
# FONCTIONS DE CONFIGURATION (autonomes, sans import externe)
# =============================================================================

from features.raf.utils.config import Config, deep_merge, flatten_dict, build_config

# =============================================================================
# CHARGEMENT DE LA CONFIGURATION
# =============================================================================

config = build_config(project_root=project_root, environment=ENV)

# =============================================================================
# AFFICHAGE DU RÉSUMÉ
# =============================================================================

print("\n" + "=" * 60)
print("✅ CONFIGURATION PRÊTE")
print("=" * 60)
print(f"📂 Projet: {config.project_root}")
print(f"📊 Dataset: {config.data_dir}")
print(f"🏷️ Classes: {', '.join(config.classes)}")
print(f"🎛️ Images: {config.img_size}")
print(f"🔧 Batch: {config.batch_size} | Époques: {config.epochs}")
print(f"📐 Dataset accessible: {'✅' if config.data_dir.exists() else '❌'}")
print("=" * 60)
print("\n💡 Variables disponibles:")
print("   • config: Configuration du projet (Config)")
print("   • ENV: Environnement ('colab', 'wsl', 'local')")
print("\n🎯 Imports disponibles:")
print("   • from features.raf.data import ...")
print("   • from features.raf.interpretability import ...")
print("   • from features.raf.models import ...")
print("=" * 60)


🌍 Environnement: WSL
⚠️ Erreur import RAF: cannot import name 'get_config' from 'features.raf.utils' (/home/cepa/DST/projet_DS/DS_COVID/src/features/raf/utils/__init__.py)
💡 Certains modules peuvent ne pas être disponibles

✅ CONFIGURATION PRÊTE
📂 Projet: /home/cepa/DST/projet_DS/DS_COVID
📊 Dataset: /home/cepa/DST/projet_DS/DS_COVID/data/raw/COVID-19_Radiography_Dataset/COVID-19_Radiography_Dataset
🏷️ Classes: COVID, Lung_Opacity, Normal, Viral Pneumonia
🎛️ Images: (256, 256)
🔧 Batch: 32 | Époques: 50
📐 Dataset accessible: ✅

💡 Variables disponibles:
   • config: Configuration du projet (Config)
   • ENV: Environnement ('colab', 'wsl', 'local')

🎯 Imports disponibles:
   • from features.raf.data import ...
   • from features.raf.interpretability import ...
   • from features.raf.models import ...


# 🔍 Interprétabilité des Modèles COVID-19 avec SHAP, GradCAM et LIME

Ce notebook démontre l'utilisation des techniques d'interprétabilité **SHAP**, **GradCAM** et **LIME** pour expliquer les prédictions des modèles de classification COVID-19.

## 📋 Objectifs

1. **🎯 GradCAM** : Visualiser les zones importantes pour les prédictions CNN
2. **📊 SHAP** : Quantifier l'importance des features pour tous types de modèles
3. **🍃 LIME** : Explications locales par superpixels pour modèles image
4. **⚖️ Comparaison** : Analyser la cohérence entre les différentes méthodes
5. **🏥 Application** : Cas d'usage médical pour le diagnostic COVID-19

## 🚀 Framework RAF

Nous utilisons le code situé dans `raf/` qui intègre maintenant les modules d'interprétabilité :
- `raf.interpretability.SHAPExplainer` - Explications SHAP universelles
- `raf.interpretability.GradCAMExplainer` - Visualisations GradCAM pour CNN
- `raf.interpretability.LIMEExplainer` - Explications LIME par superpixels
- `raf.interpretability.InterpretabilityAnalyzer` - Analyse comparative
- `raf.data.DataLoader` - Chargement de vos vraies données COVID-19

## ⚙️ Configuration et Imports

In [3]:
# Imports standards
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns
from tqdm.notebook import tqdm
import time
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
import traceback
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

# Configuration graphiques depuis la config JSON
plt.style.use(config.plot_style)
sns.set_palette(config.color_palette)
plt.rcParams['figure.figsize'] = config.figure_size
plt.rcParams['figure.dpi'] = config.dpi

print("✅ Imports et configuration graphique terminés")

2025-11-04 10:06:28.908040: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✅ Imports et configuration graphique terminés


## 📊 Chargement des Données et Modèles

Nous allons utiliser des données simulées pour la démonstration. En pratique, vous remplacerez ceci par vos vrais modèles et données.

In [4]:
# Initialisation du DataLoader RAF
print("\n🔧 Initialisation du DataLoader...")

# Import du DataLoader depuis RAF
from features.raf.data_loading import DataLoader

# Configuration du nombre max d'images par classe (pour tests rapides)
config.max_images_per_class = 100  # Ajuster selon vos besoins

# Créer le DataLoader
loader = DataLoader(config=config)

print("✅ DataLoader configuré avec succès")


🔧 Initialisation du DataLoader...


ImportError: cannot import name 'get_config' from 'features.raf.utils.config' (/home/cepa/DST/projet_DS/DS_COVID/src/features/raf/utils/config.py)